Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: 요청/응답 자동 로깅 테이블로부터 사용자별 요청 횟수와 상세 토큰 소모량을 집계 및 모니터링한다.

## 제미나이 API 계정별 사용량 모니터링 (Gemini API Usage by Account )

### 1. 의존성 패키지 설치

빅쿼리 데이터 적재 및 쿼리를 정밀 수행하기 위해 필요한 핵심 클라우드 라이브러리를 설치한다.

In [ ]:
!pip install --quiet google-cloud-bigquery

### 2. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 활성화되어 있는 사용자의 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색하고 전역 변수를 선언한다.

In [ ]:
import google.auth

BIGQUERY_DATASET_ID = "gcp_logs"
DAYS = 7
TABLE_NAME = "request_response_logging"

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print(f"[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id" # 본인의 실제 GCP 프로젝트 ID로 변경하기 바란다.

### 3. 빅쿼리 데이터세트 및 로깅 테이블 존재 여부 검증

로그가 수집되는 빅쿼리 데이터세트와 요청/응답 로깅 테이블의 존재 여부를 미리 확인한다.

In [ ]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

bq_client = bigquery.Client(project=project_id)

# 데이터세트 존재 여부 확인 및 생성
try:
  bq_client.get_dataset(BIGQUERY_DATASET_ID)
  print(f"[통과] 빅쿼리 데이터세트 '{BIGQUERY_DATASET_ID}'가 이미 존재한다.")
except NotFound:
  print(f"[안내] 빅쿼리 데이터세트 '{BIGQUERY_DATASET_ID}'가 존재하지 않아 새로 생성한다...")
  dataset_ref = bq_client.dataset(BIGQUERY_DATASET_ID)
  dataset = bigquery.Dataset(dataset_ref)
  dataset.location = "asia-northeast3"  # 기본값 서울 리전
  bq_client.create_dataset(dataset)

# 요청/응답 로깅 테이블 존재 여부 확인
table_ref = f"{project_id}.{BIGQUERY_DATASET_ID}.{TABLE_NAME}"
try:
  bq_client.get_table(table_ref)
  print(f"[통과] 로깅 테이블 '{TABLE_NAME}'이 존재함을 확인했다.")
except NotFound:
  print(f"[오류] '{TABLE_NAME}' 테이블이 존재하지 않는다.")
  print("       이 오류를 해결하려면 먼저 'gemini-api-request-response-logging.ipynb' 노트북을 구동하여")
  print("       파운데이션 모델의 자동 로깅 설정을 활성화하고 API를 호출하기 바란다.")
  raise FileNotFoundError(f"'{TABLE_NAME}' 테이블이 존재하지 않아 조회를 중단한다.")

### 4. 빅쿼리 SQL 기반 계정별 토큰 사용량 및 호출 통계 집계

빅쿼리 요청/응답 로깅 테이블을 조회하여 최근 DAYS일 동안의 사용자별 호출 통계를 분석한다.

**빅쿼리 비용 최적화 설계:**
* **과금 방지용 파티션 필터**: 대량의 데이터 스캔으로 인한 불필요한 빅쿼리 분석 비용 폭증을 방지하기 위해, 파티션 컬럼인 `logging_time` 필터 조건절을 WHERE 조건 내에 명시하여 최근 DAYS일 동안의 로그만 강제 제한 스캔하도록 설계(Partition Pruning )한다.

In [ ]:
query = f"""
  SELECT COALESCE(JSON_EXTRACT_SCALAR(full_request, '$.labels.principal_email'), 'default_user') AS user_email,
         COUNT(1) AS request_count,
         SUM(LAX_INT64(full_response.usageMetadata.promptTokenCount)) AS total_input_tokens,
         SUM(LAX_INT64(full_response.usageMetadata.candidatesTokenCount)) AS total_output_tokens,
         SUM(LAX_INT64(full_response.usageMetadata.thoughtsTokenCount)) AS total_thinking_tokens
    FROM `{project_id}.{BIGQUERY_DATASET_ID}.{TABLE_NAME}`
   WHERE logging_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {DAYS} DAY)
   GROUP BY user_email
   ORDER BY request_count DESC;
"""

try:
  query_job = bq_client.query(query)
  results = query_job.result()
  
  print("\n=== [사용자별 제미나이 API 호출 통계] ===")
  for row in results:
    print(f"사용자 계정: {row.user_email}")
    print(f"  - 호출 횟수: {row.request_count}회")
    print(f"  - 프롬프트 토큰 합계: {row.total_input_tokens}")
    print(f"  - 답변 생성 토큰 합계: {row.total_output_tokens}")
    print(f"  - 싱킹 토큰 합계: {row.total_thinking_tokens}")
    print("-"*48)
except Exception as e:
  print(f"[오류] 빅쿼리 로그를 조회하는 도중 에러가 발생했다: {e}")